<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Yesterday predicts today

**[Yesterday predicts today](https://learning.nextia-ai.com/courses/forecasting/m02/yesterday-predicts-today/)** · Forecasting and Time Series: Predict What Comes Next · Module 2, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build lag columns, measure the memory of a series with the autocorrelation function (ACF), and see why a random split flatters a forecast.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The parts of a series, from [Trend, season and noise](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/yesterday-predicts-today/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M02-L02-yesterday-predicts-today/yesterday-predicts-today-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny series: a bakery's loaves

A bakery sells loaves for 8 days. A **lag** is the same series moved down by some steps: the lag-1 column holds yesterday's value next to today's.

The **autocorrelation** at lag 1 measures how much "above or below the average" today follows yesterday. Subtract the mean from every value, multiply each day's distance by yesterday's distance, add up the products, and divide by the sum of the squared distances.

> **Predict.** The loaves rise slowly over the 8 days. Will the autocorrelation at lag 1 be positive or negative? Then run the cell.

In [ ]:
loaves = pd.DataFrame({"day": range(1, 9), "loaves": [20, 23, 25, 24, 28, 30, 27, 31]})
mean = loaves["loaves"].mean()
loaves["distance"] = loaves["loaves"] - mean
loaves["yesterday"] = loaves["loaves"].shift(1)          # the lag-1 column
loaves["product"] = loaves["distance"] * loaves["distance"].shift(1)
print(loaves.to_string(index=False))
print("mean", mean, "| sum of products", loaves["product"].sum(), "| sum of squares", (loaves["distance"] ** 2).sum())
print("autocorrelation at lag 1:", loaves["product"].sum() / (loaves["distance"] ** 2).sum())

> **Check.** You should see `mean 26.0 | sum of products 36.0 | sum of squares 96.0` and `autocorrelation at lag 1: 0.375`.

statsmodels' `acf()` does the same computation for every lag at once. Run the next cell: its lag-1 value is the same 0.375.

In [ ]:
from statsmodels.tsa.stattools import acf

print(acf(loaves["loaves"], nlags=1).round(3))

> **Check.** You should see `[1.    0.375]`. Lag 0 is always 1: a series is perfectly correlated with itself.

## 2. Larkfield's orders: today against yesterday

The next cell reads the daily orders (the missing day filled, as in the last lesson), adds the lag-1 and lag-7 columns, and draws today against yesterday.

In [ ]:
import matplotlib.pyplot as plt

orders = pd.read_csv("data/orders.csv", parse_dates=["date"]).groupby("date")["orders"].sum().asfreq("D")
orders = orders.fillna((orders.shift(7) + orders.shift(-7)) / 2).round()
lags = pd.DataFrame({"today": orders, "lag_1": orders.shift(1), "lag_7": orders.shift(7)})
print(lags.head(8).to_string())
lags.plot.scatter(x="lag_1", y="today", s=6, figsize=(4.5, 4.5), xlabel="Orders yesterday", ylabel="Orders today")
plt.show()

> **Check.** You should see 8 rows: `lag_1` is empty (`NaN`) on the first day and `lag_7` on the first seven days, because there is no earlier day. The dots of the chart lie along a rising band: a busy yesterday usually means a busy today.

## 3. The autocorrelation function (ACF)

The ACF is the autocorrelation at every lag, here from 0 to 28 days.

> **Predict.** Which lags between 1 and 28 will stand out above their neighbours? Then run the cell.

In [ ]:
acf_orders = acf(orders, nlags=28)
print({k: round(float(acf_orders[k]), 3) for k in (1, 2, 7, 14, 21, 28)})
plt.figure(figsize=(9, 3))
plt.bar(range(29), acf_orders, color=["#0054D8" if k % 7 == 0 else "#9AA8BF" for k in range(29)])
plt.xlabel("Lag (days)"); plt.ylabel("Autocorrelation"); plt.xticks(range(0, 29, 7))
plt.show()

> **Check.** You should see `{1: 0.827, 2: 0.766, 7: 0.674, 14: 0.571, 21: 0.479, 28: 0.426}`. The bars slowly fall with the lag, and lags 7, 14, 21 and 28 are higher than their neighbours: the weekly season.

The next cell does the same for the hourly help-desk tickets, up to one week (168 hours).

In [ ]:
tickets = pd.read_csv("data/tickets_hourly.csv.gz", parse_dates=["timestamp"]).groupby("timestamp")["tickets"].sum()
acf_tickets = acf(tickets.to_numpy(dtype=float), nlags=168)
print({k: round(float(acf_tickets[k]), 3) for k in (1, 12, 24, 168)})
plt.figure(figsize=(9, 3))
plt.plot(range(169), acf_tickets)
plt.xlabel("Lag (hours)"); plt.ylabel("Autocorrelation"); plt.xticks(range(0, 169, 24))
plt.show()

> **Check.** You should see `{1: 0.822, 12: -0.438, 24: 0.81, 168: 0.832}`. The same hour yesterday (lag 24) is almost as close as the hour before, and the same hour last week (lag 168) is the closest of all. Twelve hours away, the night against the day, the value is negative.

## 4. Your turn: the parasol

> **Your turn.** Compute the ACF of the parasol's daily sales (P025, the three regions added up) with `acf(..., nlags=28)`. Use `load_sales(["P025"])["y"]`. Put the value at lag 7, rounded to 3 decimals, in `parasol_lag7` as a Python float. Then run the check cell.

In [ ]:
parasol = load_sales(["P025"])["y"]
acf_parasol = acf(parasol, nlags=28)
parasol_lag7 = round(float(acf_parasol[7]), 3)
print({k: round(float(acf_parasol[k]), 3) for k in (1, 7, 14, 28)})

In [ ]:
expect('parasol_lag7', parasol_lag7, 0.675)

The parasol's ACF at lag 1 is 0.827, the same as the orders, but it falls more slowly. Its high values come from the yearly season: a summer day sits among summer days, and a winter day among winter days.

## 5. Failure case: a random split lets the neighbours tell the answer

Tomás tests a simple rule: "forecast a day with the mean of the nearest days that we know, one before and one after". He takes 20% of the days at random as the test set. Then he tests the same rule as a real forecast: at each of the course's 13 forecast days, he knows only the past and forecasts the next 14 days.

> **Predict.** Will the rule's error be smaller with the random split or with the real forecasts? Then run the cell.

In [ ]:
y = orders.to_numpy()

def nearest_known(i, known):
    """The mean of the nearest known day before day i and the nearest known day after it (if there is one)."""
    before, after = known[known < i], known[known > i]
    return np.mean(([y[before[-1]]] if len(before) else []) + ([y[after[0]]] if len(after) else []))

rng = np.random.default_rng(0)
test = np.sort(rng.choice(len(y), size=len(y) // 5, replace=False))
known = np.setdiff1d(np.arange(len(y)), test)
random_mae = np.mean([abs(nearest_known(i, known) - y[i]) for i in test])

errors = []
for origin in ORIGINS:
    i0 = orders.index.get_loc(origin)
    past = np.arange(i0 + 1)                  # only the days up to the forecast day
    errors += [abs(nearest_known(i, past) - y[i]) for i in range(i0 + 1, i0 + 15)]
time_mae = np.mean(errors)
print(f"random split: {len(test)} test days, MAE {random_mae:.1f} orders")
print(f"real forecasts: {len(errors)} days, MAE {time_mae:.1f} orders")

> **Check.** You should see `random split: 145 test days, MAE 59.3 orders` and `real forecasts: 182 days, MAE 106.8 orders`.

The rule is the same. With the random split, almost every test day has its yesterday and its tomorrow in the known days, and the ACF says that they are close to it (0.827). The rule copies the answer from the neighbours. In a real forecast, tomorrow is never known. The random split makes the error look 45% smaller than it is. A forecast must be tested the way it is used: from the past only. Module 3 builds that test.

## Next

You can now measure the memory of a series and explain why a random split flatters a forecast. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/yesterday-predicts-today/) has the charts and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Calendars, holidays and events](https://learning.nextia-ai.com/courses/forecasting/m02/calendars-holidays-and-events/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m02/yesterday-predicts-today/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.